# agents

> This module specifies the Agent, Policy, and RL Classes. An agent carries one or more policies when exploring an environment, and updates parameters using reinforcement learning.

In [ ]:
#| default_exp agents

In [ ]:
#| export
import math
from dataclasses import dataclass
import numpy as np


In [ ]:
#| hide
from nbdev.showdoc import *
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
class Policy:
    '''This Policy Class specifies a general architecture. '''
    def __init__(self):
        self.theta = None
        self.actions = None
        self.pi = None

In [ ]:
#| export
class flexiblePolicy(Policy):
    '''This flexiblePolicy Class specifies a fully flexible policy with a set of basis functions
    and a learnable set of weights. '''
    def __init__(self, omega=None, fk=None, wParams=None, integ=0.0):
        super().__init__()
        self.omega = np.asarray(omega) if omega is not None else None
        self.fk = np.asarray(fk) if fk is not None else None
        self.wParams = wParams
        self.integ = float(integ)

In [ ]:
#| export
class RL:
    '''This RL Class performs reinforcement learning: after taking an action, update parameters
    based on feedback from the environment.
    '''
    def __init__(self,policy):
        self.policy = policy

    def chooseAction(self):
        raise NotImplementedError

    def updateParams(self):
        raise NotImplementedError

In [ ]:
#| export
class Agent:
    '''This Class specifies agents with a defined policy and reinforcement learning algorithm. '''
    def __init__(self,theta,goal,policy,rl,params):
        self.theta = theta   # current heading (radians)
        self.goal = goal
        self.policy = policy
        self.rl = rl
        self.params = params  # structure containing simulation parameters
        self.dx = None  # change in heading
        self.dt = None  # change in time

    def samplePolicy(self):
        raise NotImplementedError("Override me")

In [ ]:
#| export
@dataclass
class Step:
    "One policy sample: a 1-sample fixation or a saccade lasting `dtS` samples."
    theta: float      # heading before the step, radians
    ind: int          # basis column used for this step
    xi: float         # integrator before the step
    dxi: float        # integrator change drawn this step
    saccade: bool
    direction: int    # +1 CW, -1 CCW, 0 for a fixation
    dx: float         # heading change in radians (0.0 for a fixation)
    dt: int           # duration in samples (1 or dtS)


In [ ]:
#| export
class flexibleAgent(Agent):
    "Agent whose fixations and saccades depend on heading through a `flexiblePolicy` (paper Algorithm 2)."
    def __init__(self, theta, goal, policy, rl, params, rng=None):
        if not isinstance(policy, flexiblePolicy):
            raise TypeError("policy must be a flexiblePolicy instance")
        if policy.omega is None or policy.fk is None or policy.wParams is None:
            raise ValueError("policy needs omega, fk and wParams")
        nb = policy.omega.size // 2
        if policy.omega.size != 2 * nb:
            raise ValueError("omega must have even length")
        if policy.fk.shape != (nb, params.nx):
            raise ValueError(f"fk must have shape (omega.size // 2, params.nx) = {(nb, params.nx)}")
        if len(policy.wParams) != 6:
            raise ValueError("wParams needs exactly 6 values: (k, fmax, offset) for drift, then for turn bias")
        super().__init__(float(theta) % (2 * np.pi), goal, policy, rl, params)
        self.rng = rng if rng is not None else np.random.default_rng()

    def _heading_index(self, nx):
        "Index of the basis column nearest the current heading (wrapped angular distance)."
        centers = np.linspace(0, 2 * np.pi, num=nx, endpoint=False)
        d = np.abs((self.theta - centers + np.pi) % (2 * np.pi) - np.pi)
        return int(np.argmin(d))

    def samplePolicy(self):
        "Sample one fixation sample or one saccade and return it as a `Step`; the integrator lives on the policy."
        pol, p = self.policy, self.params
        ind = self._heading_index(p.nx)
        nu, pR = self.convertParams(pol.omega, pol.fk[:, ind], pol.wParams)
        xi0 = pol.integ
        dxi = float(self.rng.normal(nu * p.delT, p.sigF * np.sqrt(p.delT)))
        if xi0 + dxi > p.aF:
            direction = 1 if self.flipCoin(pR) else -1
            self.dx = direction * float(np.deg2rad(self.rng.lognormal(p.muS, p.sigmaS)))
            self.dt = int(p.dtS)
            pol.integ = 0.0
        else:
            direction, self.dx, self.dt = 0, 0.0, 1
            pol.integ = xi0 + dxi
        return Step(self.theta, ind, xi0, dxi, direction != 0, direction, self.dx, self.dt)

    def convertParams(self, omega, fk_col, w_params):
        "Converts weights and one basis column into a drift rate `nu` and a rightward-saccade probability `pR`."
        omega, fk_col = np.asarray(omega), np.asarray(fk_col)
        nb = omega.size // 2
        if omega.size != 2 * nb:
            raise ValueError("omega must have even length")
        if fk_col.size != nb:
            raise ValueError("fk column length must match omega.size // 2")
        nu_f = self.sigmoid(omega[:nb] @ fk_col, w_params[0], w_params[1], w_params[2])
        p_r = self.sigmoid(omega[nb:] @ fk_col, w_params[3], w_params[4], w_params[5])
        return float(nu_f), float(p_r)

    def sigmoid(self, x, k, fmax, offset):
        "fmax / (1 + exp(-k x)) - offset (paper Eq. 17)."
        return np.divide(fmax, (1 + np.exp(-k * x))) - offset

    def sigmoidGrad(self, x, k, fmax):
        "Derivative of `sigmoid` with respect to x."
        s = 1.0 / (1.0 + np.exp(-k * x))
        return fmax * k * s * (1.0 - s)

    def flipCoin(self, p):
        "True with probability p."
        return self.rng.random() < p


In [ ]:
#| export
class PolicyGradientRL(RL):
    "Online policy-gradient learning for `flexibleAgent` (paper Eqs. 13-18, Algorithm 1)."
    def __init__(self, policy, lr=1.0, eps=1e-12):
        super().__init__(policy)
        self.lr, self.eps = float(lr), float(eps)

    def chooseAction(self, agent):
        "Sample the next step from the agent's policy."
        return agent.samplePolicy()

    def updateParams(self, agent, step, reward):
        "omega <- omega + lr * reward * grad log pi(step)."
        self.policy.omega = self.policy.omega + self.lr * reward * self.gradLogPi(agent, step)

    def _terms(self, agent, step, omega):
        "Quantities shared by `logPi` and `gradLogPi`, with probabilities clamped away from 0 and 1."
        p, w = agent.params, agent.policy.wParams
        g = agent.policy.fk[:, step.ind]
        nb = g.size
        xF, xS = float(omega[:nb] @ g), float(omega[nb:] @ g)
        nu = agent.sigmoid(xF, w[0], w[1], w[2])
        dS = min(1 - self.eps, max(self.eps, agent.sigmoid(xS, w[3], w[4], w[5])))
        s = p.sigF * math.sqrt(p.delT)
        z = (p.aF - (step.xi + nu * p.delT)) / s
        PF = min(1 - self.eps, max(self.eps, 0.5 * (1 + math.erf(z / math.sqrt(2)))))
        return g, xF, xS, dS, s, z, PF

    def logPi(self, agent, step, omega=None):
        "log-probability of `step` under weights `omega` (default: current weights)."
        omega = self.policy.omega if omega is None else np.asarray(omega)
        _, _, _, dS, _, _, PF = self._terms(agent, step, omega)
        if not step.saccade:
            return math.log(PF)
        return math.log(1 - PF) + math.log(dS if step.direction > 0 else 1 - dS)

    def gradLogPi(self, agent, step):
        "Gradient of `logPi` with respect to all 2*nb weights (fixation weights first)."
        p, w = agent.params, agent.policy.wParams
        g, xF, xS, dS, s, z, PF = self._terms(agent, step, self.policy.omega)
        phi = math.exp(-z * z / 2) / (s * math.sqrt(2 * math.pi))
        gNu = agent.sigmoidGrad(xF, w[0], w[1]) * g
        if not step.saccade:
            return np.concatenate([-p.delT * phi * gNu / PF, np.zeros(g.size)])
        gDS = agent.sigmoidGrad(xS, w[3], w[4]) * g
        gS = gDS / dS if step.direction > 0 else -gDS / (1 - dS)
        return np.concatenate([p.delT * phi * gNu / (1 - PF), gS])


Tests use a small agent with a flat basis so results are easy to predict:


In [ ]:
from fastcore.test import test_eq, test_fail, test_close
from explRL.params import Params

def toy_agent(theta=0.0, aF=10.0, nb=4, nx=8, omega=None, seed=None):
    "Builds a small flexibleAgent for testing."
    policy = flexiblePolicy(omega=np.zeros(2 * nb) if omega is None else omega,
                            fk=np.ones((nb, nx)), wParams=np.array([0.5, 1.0, 0.0, 0.5, 1.0, 0.0]))
    params = Params(nx=nx, nb=nb, delT=0.01, sigF=0.1, aF=aF, muS=0.0, sigmaS=0.5, tS=0.05)
    return flexibleAgent(theta, None, policy, PolicyGradientRL(policy), params, rng=np.random.default_rng(seed))

agent = toy_agent()
nu, pR = agent.convertParams(agent.policy.omega, agent.policy.fk[:, 0], agent.policy.wParams)
test_eq((nu, pR), (0.5, 0.5))   # all-zero weights give sigmoid(0) = fmax/2


With a high threshold the agent fixates; with a tiny threshold it saccades, the step lasts `dtS` and the integrator resets:


In [ ]:
agent = toy_agent(aF=10.0, seed=1)
step = agent.samplePolicy()
test_eq((step.saccade, step.dx, step.dt, step.direction), (False, 0.0, 1, 0))
test_eq(step.ind, agent._heading_index(8))
test_close(agent.policy.integ, step.xi + step.dxi)

agent = toy_agent(aF=1e-6, seed=2)
step = agent.samplePolicy()
assert step.saccade and step.direction in (1, -1)
test_eq(step.dt, agent.params.dtS)
test_eq(agent.policy.integ, 0.0)
assert isinstance(step.dx, float) and step.dx != round(step.dx)   # radians, not rounded
assert np.sign(step.dx) == step.direction


Headings are stored wrapped into [0, 2π), and the nearest basis column wraps too:


In [ ]:
test_close(toy_agent(theta=-0.1).theta, 2 * np.pi - 0.1)
test_eq(toy_agent(theta=2 * np.pi - 0.01)._heading_index(8), 0)
test_eq(toy_agent(theta=np.pi)._heading_index(8), 4)
test_eq(toy_agent(theta=7 * np.pi)._heading_index(8), 4)


The same seed gives the same sequence of steps:


In [ ]:
a0, a1 = toy_agent(aF=0.05, seed=99), toy_agent(aF=0.05, seed=99)
for _ in range(20):
    test_eq(a0.samplePolicy(), a1.samplePolicy())


`sigmoidGrad` and `gradLogPi` match finite differences, for a fixation and for clockwise and counter-clockwise saccades (AC-2):


In [ ]:
ag = toy_agent(omega=np.linspace(-0.3, 0.3, 8))
x = 0.3
test_close(ag.sigmoidGrad(x, 1.0, 1.0), (ag.sigmoid(x + 1e-5, 1, 1, 0) - ag.sigmoid(x - 1e-5, 1, 1, 0)) / 2e-5, eps=1e-6)

def fd_check(step, h=1e-6):
    rl, om = ag.rl, ag.policy.omega.copy()
    g = rl.gradLogPi(ag, step)
    fd = np.array([(rl.logPi(ag, step, om + h * e) - rl.logPi(ag, step, om - h * e)) / (2 * h) for e in np.eye(om.size)])
    assert np.max(np.abs(g - fd)) <= 1e-4 * max(1.0, np.max(np.abs(fd))), (g, fd)

xi = ag.params.aF - 0.02   # near the threshold, so fixation and saccade are both plausible
fd_check(Step(0.0, 0, xi, 0.0, False, 0, 0.0, 1))
fd_check(Step(0.0, 0, xi, 0.0, True, 1, 0.5, ag.params.dtS))
fd_check(Step(0.0, 0, xi, 0.0, True, -1, -0.5, ag.params.dtS))


Gradients stay finite when the fixation probability is very close to 0 or 1 (NFR-7):


In [ ]:
for xi in (-50.0, 50.0):
    for sac, d in ((False, 0), (True, 1), (True, -1)):
        assert np.isfinite(ag.rl.gradLogPi(ag, Step(0.0, 0, xi, 0.0, sac, d, 0.0, 1))).all()


A positive reward after a clockwise saccade raises `pR` at that heading, a negative reward lowers it, and zero reward changes nothing (AC-3):


In [ ]:
def pR_after(reward):
    a = toy_agent()
    step = Step(0.0, 0, 0.0, 0.0, True, 1, 0.5, a.params.dtS)
    a.rl.updateParams(a, step, reward)
    return a.convertParams(a.policy.omega, a.policy.fk[:, 0], a.policy.wParams)[1], a.policy.omega

p_pos, _ = pR_after(1.0)
p_neg, _ = pR_after(-1.0)
assert p_pos > 0.5 > p_neg
_, om0 = pR_after(0.0)
test_eq(om0, np.zeros(8))


Bad inputs fail early, and the base `RL` class still has to be subclassed:


In [ ]:
test_fail(lambda: toy_agent(omega=np.zeros(7)), contains="even length")
test_fail(lambda: flexibleAgent(0.0, None, Policy(), None, Params(nx=8, nb=4)), contains="flexiblePolicy")
test_fail(lambda: RL(None).chooseAction())
test_fail(lambda: RL(None).updateParams())


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
